# Démonstration du Transport (Advection + Diffusion)

Ce notebook démontre l'utilisation du module `seapopym.transport` pour simuler le transport d'une biomasse passive.
Le modèle est configuré via le Blueprint avec deux processus : advection et diffusion.

In [ ]:
from datetime import timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import Markdown

from seapopym.blueprint import Blueprint
from seapopym.controller import SimulationConfig, SimulationController
from seapopym.standard.coordinates import Coordinates
from seapopym.transport import (
    BoundaryConditions,
    BoundaryType,
    check_diffusion_stability,
    compute_advection_tendency,
    compute_diffusion_tendency,
    compute_spherical_cell_areas,
    compute_spherical_dx,
    compute_spherical_dy,
    compute_spherical_face_areas_ew,
    compute_spherical_face_areas_ns,
)

## 1. Configuration de la Simulation

In [ ]:
config = SimulationConfig(
    start_date="2020-01-01",
    end_date="2020-01-10",
    timestep=timedelta(hours=1),
)

## 2. Création de l'Environnement (Grille et Forçages)

In [ ]:
# Grille spatiale (10x10 degrés)
lats = np.linspace(0, 10, 20)
lons = np.linspace(0, 10, 20)
times = np.arange(
    config.start_date,
    pd.to_datetime(config.end_date) + config.timestep,
    config.timestep,
    dtype="datetime64[ns]",
)

# Courants (Forçages)
# Champ de vitesse uniforme vers l'Est (u=0.5 m/s, v=0)
u_data = np.ones((len(times), len(lats), len(lons))) * 1.5
v_data = np.zeros((len(times), len(lats), len(lons)))

# Diffusivité (Forçage)
D_data = np.ones((len(times), len(lats), len(lons))) * 5000.0  # 5000 m²/s

forcings = xr.Dataset(
    {
        "current_u": ((Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value), u_data),
        "current_v": ((Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value), v_data),
        "diffusivity": ((Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value), D_data),
    },
    coords={
        Coordinates.T.value: times,
        Coordinates.Y.value: lats,
        Coordinates.X.value: lons,
    },
)

# Compute geometric quantities using grid module
# These are static (not time-varying)
lats_da = xr.DataArray(lats, dims=[Coordinates.Y.value])
lons_da = xr.DataArray(lons, dims=[Coordinates.X.value])

cell_areas = compute_spherical_cell_areas(lats_da, lons_da)
face_areas_ew = compute_spherical_face_areas_ew(lats_da, lons_da)
face_areas_ns = compute_spherical_face_areas_ns(lats_da, lons_da)
dx = compute_spherical_dx(lats_da, lons_da)
dy = compute_spherical_dy(lats_da, lons_da)

# Add geometric quantities as static forcings (no time dimension)
# We'll drop the lon_face and lat_face dimensions for now (approximation)
grid_params = xr.Dataset(
    {
        "cell_areas": cell_areas,
        "dx": dx,
        "dy": dy,
    },
    coords={Coordinates.Y.value: lats, Coordinates.X.value: lons},
)

# Note: face_areas have different dimensions (lat, lon+1) and (lat+1, lon)
# For simplicity in Blueprint, we'll use cell-centered approximations
# A more sophisticated approach would handle staggered grids properly

# Fusionner pour le contrôleur
forcings = xr.merge([forcings, grid_params])

# Define boundary conditions (closed poles, periodic longitude)
boundary_conditions = BoundaryConditions(
    north=BoundaryType.CLOSED,
    south=BoundaryType.CLOSED,
    east=BoundaryType.CLOSED,
    west=BoundaryType.CLOSED,
)

print("Geometric quantities computed:")
print(f"  Cell areas: {cell_areas.shape}")
print(f"  Face areas E/W: {face_areas_ew.shape}")
print(f"  Face areas N/S: {face_areas_ns.shape}")
print(f"  dx: {dx.shape}")
print(f"  dy: {dy.shape}")
print(f"\nBoundary conditions: {boundary_conditions}")

## 2.5 Vérification de la Stabilité

In [ ]:
# Check diffusion stability
D_max = D_data.max()
dt_seconds = config.timestep.total_seconds()

stability_info = check_diffusion_stability(D=D_max, dx=dx, dy=dy, dt=dt_seconds)

print("=== Diffusion Stability Check ===")
print(f"Stable: {stability_info['is_stable']}")
print(f"Current dt: {dt_seconds:.1f} s ({dt_seconds / 3600:.2f} hours)")
print(
    f"Maximum stable dt: {stability_info['dt_max']:.1f} s ({stability_info['dt_max'] / 3600:.2f} hours)"
)
print(f"CFL number: {stability_info['cfl_diffusion']:.4f} (should be ≤ 0.25)")
print(f"Safety margin: {stability_info['margin']:.2f}x")
print(f"D_max: {stability_info['D_max']:.1f} m²/s")
print(f"dx_min: {stability_info['dx_min']:.1f} m")
print(f"dy_min: {stability_info['dy_min']:.1f} m")

if not stability_info["is_stable"]:
    print("\n⚠️  WARNING: Simulation is UNSTABLE! Reduce timestep.")
elif stability_info["cfl_diffusion"] > 0.2:
    print("\n⚠️  WARNING: Close to stability limit. Consider reducing timestep.")
else:
    print("\n✓ Stability criterion satisfied.")

## 3. État Initial (Biomasse)

In [ ]:
# Initialisation d'un "blob" de biomasse au centre
biomass_data = np.zeros((len(lats), len(lons)))
center_y, center_x = len(lats) // 2, len(lons) // 2
biomass_data[center_y - 2 : center_y + 2, center_x - 2 : center_x + 2] = 100.0

initial_state = xr.Dataset(
    {
        "Plankton/biomass": ((Coordinates.Y.value, Coordinates.X.value), biomass_data),
        "dt": config.timestep.total_seconds(),
    },
    coords={Coordinates.Y.value: lats, Coordinates.X.value: lons},
)

# Optional: Add a land/ocean mask (1=ocean, 0=land)
# For this demo, we use all ocean (no land)
# mask = xr.DataArray(np.ones((len(lats), len(lons))), dims=[Coordinates.Y.value, Coordinates.X.value])
# initial_state["mask"] = mask

plt.figure(figsize=(6, 5))
initial_state["Plankton/biomass"].plot()
plt.title("Biomasse Initiale")
plt.show()

In [ ]:
def configure_transport_model(bp: Blueprint):
    # 1. Enregistrement des Forçages
    bp.register_forcing(
        "current_u",
        dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value),
        units="m/s",
    )
    bp.register_forcing(
        "current_v",
        dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value),
        units="m/s",
    )
    bp.register_forcing(
        "diffusivity",
        dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value),
        units="m**2/s",
    )
    bp.register_forcing("cell_areas", dims=(Coordinates.Y.value, Coordinates.X.value), units="m**2")
    bp.register_forcing("dx", dims=(Coordinates.Y.value, Coordinates.X.value), units="m")
    bp.register_forcing("dy", dims=(Coordinates.Y.value, Coordinates.X.value), units="m")

    # 2. Create wrapper functions that capture boundary_conditions and face_areas
    # These wrappers have Blueprint-compatible signatures (all parameters are DataArrays)

    def advection_wrapper(state, u, v, cell_areas):
        """Wrapper for advection that captures boundary_conditions and face_areas."""
        # For simplicity, we approximate face_areas with cell-centered values
        # A more sophisticated approach would use staggered grids
        return compute_advection_tendency(
            state=state,
            u=u,
            v=v,
            cell_areas=cell_areas,
            face_areas_ew=face_areas_ew,  # Captured from outer scope
            face_areas_ns=face_areas_ns,  # Captured from outer scope
            boundary_conditions=boundary_conditions,  # Captured from outer scope
            mask=None,  # No land mask for this demo
        )

    def diffusion_wrapper(state, D, dx, dy):
        """Wrapper for diffusion that captures boundary_conditions."""
        return compute_diffusion_tendency(
            state=state,
            D=D,
            dx=dx,
            dy=dy,
            boundary_conditions=boundary_conditions,  # Captured from outer scope
            mask=None,  # No land mask for this demo
        )

    # 3. Définition du Groupe Fonctionnel "Plankton"
    bp.register_group(
        group_prefix="Plankton",
        state_variables={
            "biomass": {"dims": (Coordinates.Y.value, Coordinates.X.value), "units": "kg/m**2"}
        },
        units=[
            # Unité d'Advection
            {
                "func": advection_wrapper,
                "name": "advection",
                "input_mapping": {
                    "state": "biomass",
                    "u": "current_u",
                    "v": "current_v",
                    "cell_areas": "cell_areas",
                },
                "output_mapping": {"advection_rate": "advection_tendency"},
                "output_tendencies": {"advection_rate": "biomass"},
                "output_units": {"advection_rate": "kg/m**2/s"},
            },
            # Unité de Diffusion
            {
                "func": diffusion_wrapper,
                "name": "diffusion",
                "input_mapping": {
                    "state": "biomass",
                    "D": "diffusivity",
                    "dx": "dx",
                    "dy": "dy",
                },
                "output_mapping": {"diffusion_rate": "diffusion_tendency"},
                "output_tendencies": {"diffusion_rate": "biomass"},
                "output_units": {"diffusion_rate": "kg/m**2/s"},
            },
        ],
    )


bp = Blueprint()
configure_transport_model(bp)
Markdown("```mermaid\n" + bp.export_mermaid() + "\n```")

## 4. Configuration du Blueprint

## 5. Exécution

In [ ]:
controller = SimulationController(config, backend="sequential")
controller.setup(
    configure_transport_model, initial_state, forcings, output_variables=["Plankton/biomass"]
)

print("Running simulation...")
controller.run()
results = controller.results

## 6. Visualisation des Résultats

In [ ]:
results["Plankton/biomass"]


In [ ]:
times_to_plot

In [ ]:
# Animation simple ou plots à différents instants
nb_plot = 6
times_to_plot = np.arange(0, nb_plot * 24, 24)  # Heures

fig, axes = plt.subplots(nb_plot, figsize=(4, 3 * nb_plot))

for i, t_idx in enumerate(times_to_plot):
    if t_idx < len(results.time):
        results["Plankton/biomass"].isel(time=t_idx).plot(ax=axes[i], vmin=0, vmax=100)
        axes[i].set_title(f"T = {t_idx}h")

plt.tight_layout()
plt.show()